<a href="https://colab.research.google.com/github/pdmonika/DAA-UNIT-1-PYTHON-PROGRAMMING/blob/main/DAA_UNIT_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#EXP 1
import time
import random


def interpolation_search(arr, target):
    low, high = 0, len(arr) - 1
    comparisons = 0

    while low <= high and arr[low] <= target <= arr[high]:
        comparisons += 1

        if low == high:
            if arr[low] == target:
                return low, comparisons
            return -1, comparisons

        if arr[high] == arr[low]:
            break

        # Interpolation formula
        pos = low + int(((target - arr[low]) * (high - low)) /
                        (arr[high] - arr[low]))

        if arr[pos] == target:
            return pos, comparisons
        elif arr[pos] < target:
            low = pos + 1
        else:
            high = pos - 1

    return -1, comparisons


def binary_search(arr, target):
    low, high = 0, len(arr) - 1
    comparisons = 0

    while low <= high:
        comparisons += 1
        mid = (low + high) // 2

        if arr[mid] == target:
            return mid, comparisons
        elif arr[mid] < target:
            low = mid + 1
        else:
            high = mid - 1

    return -1, comparisons


def performance_analysis():
    sizes = [1000, 5000, 10000, 50000, 100000]

    print(f"{'Size':>10} {'IS Time(ms)':>14} {'BS Time(ms)':>14} "
          f"{'IS Comparisons':>16} {'BS Comparisons':>16}")
    print("-" * 75)

    for size in sizes:
        arr = sorted(random.sample(range(size * 10), size))
        target = arr[random.randint(0, size - 1)]

        # Interpolation Search
        start = time.perf_counter()
        for _ in range(100):
            idx_is, comp_is = interpolation_search(arr, target)
        is_time = (time.perf_counter() - start) / 100 * 1000

        # Binary Search
        start = time.perf_counter()
        for _ in range(100):
            idx_bs, comp_bs = binary_search(arr, target)
        bs_time = (time.perf_counter() - start) / 100 * 1000

        print(f"{size:>10} {is_time:>14.4f} {bs_time:>14.4f} "
              f"{comp_is:>16} {comp_bs:>16}")


# -------- Main Program --------

arr = [5, 12, 18, 25, 31, 39, 46, 54, 63, 71, 85, 92, 108, 125, 140]
target = 71

idx, comps = interpolation_search(arr, target)

print("Array:", arr)
print("Searching for:", target)
print(f"Found at index: {idx}, Comparisons: {comps}")
print()

performance_analysis()

Array: [5, 12, 18, 25, 31, 39, 46, 54, 63, 71, 85, 92, 108, 125, 140]
Searching for: 71
Found at index: 9, Comparisons: 3

      Size    IS Time(ms)    BS Time(ms)   IS Comparisons   BS Comparisons
---------------------------------------------------------------------------
      1000         0.0019         0.0020                3                8
      5000         0.0019         0.0017                4               10
     10000         0.0017         0.0021                4               13
     50000         0.0014         0.0024                3               15
    100000         0.0022         0.0023                5               14


In [ ]:
#EXP 2
import time
import random


# ---------------- Naive String Matching ----------------
def naive_search(text, pattern):
    n, m = len(text), len(pattern)
    matches = []
    comparisons = 0

    for i in range(n - m + 1):
        j = 0
        while j < m:
            comparisons += 1
            if text[i + j] != pattern[j]:
                break
            j += 1

        if j == m:
            matches.append(i)

    return matches, comparisons


# ---------------- Compute LPS Array ----------------
def compute_lps(pattern):
    m = len(pattern)
    lps = [0] * m
    length = 0
    i = 1

    while i < m:
        if pattern[i] == pattern[length]:
            length += 1
            lps[i] = length
            i += 1
        elif length != 0:
            length = lps[length - 1]
        else:
            lps[i] = 0
            i += 1

    return lps


# ---------------- KMP Algorithm ----------------
def kmp_search(text, pattern):
    n, m = len(text), len(pattern)
    lps = compute_lps(pattern)

    matches = []
    comparisons = 0

    i = 0
    j = 0

    while i < n:
        comparisons += 1

        if pattern[j] == text[i]:
            i += 1
            j += 1

        if j == m:
            matches.append(i - j)
            j = lps[j - 1]

        elif i < n and pattern[j] != text[i]:
            if j != 0:
                j = lps[j - 1]
            else:
                i += 1

    return matches, comparisons


# ---------------- Rabin-Karp Algorithm ----------------
def rabin_karp(text, pattern, q=101):
    n, m = len(text), len(pattern)
    d = 256

    h = pow(d, m - 1, q)
    p_hash = 0
    t_hash = 0

    matches = []
    comparisons = 0

    for i in range(m):
        p_hash = (d * p_hash + ord(pattern[i])) % q
        t_hash = (d * t_hash + ord(text[i])) % q

    for s in range(n - m + 1):

        if p_hash == t_hash:
            for k in range(m):
                comparisons += 1
                if text[s + k] != pattern[k]:
                    break
            else:
                matches.append(s)

        if s < n - m:
            t_hash = (d * (t_hash - ord(text[s]) * h) +
                      ord(text[s + m])) % q

            if t_hash < 0:
                t_hash += q

    return matches, comparisons


# ---------------- Main Program ----------------

text = "COMPUTERSCIENCECOMPUTERNETWORKCOMPUTERSCIENCE"
pattern = "COMPUTER"

print("Text   :", text)
print("Pattern:", pattern)

m1, c1 = naive_search(text, pattern)
m2, c2 = kmp_search(text, pattern)
m3, c3 = rabin_karp(text, pattern)

print("\nNaive Search")
print("Matches     :", m1)
print("Comparisons :", c1)

print("\nKMP Search")
print("Matches     :", m2)
print("Comparisons :", c2)

print("\nRabin-Karp Search")
print("Matches     :", m3)
print("Comparisons :", c3)


# ---------------- Performance Comparison ----------------

text_large = ''.join(random.choices('ABCD', k=10000))
patterns = ["AB", "ABC", "ABCD", "ABCDAB"]

print("\nPerformance Comparison")
print(f"{'Pattern':>10} {'Naive':>10} {'KMP':>10} {'RK':>10}")
print("-" * 45)

for p in patterns:
    _, c1 = naive_search(text_large, p)
    _, c2 = kmp_search(text_large, p)
    _, c3 = rabin_karp(text_large, p)

    print(f"{p:>10} {c1:>10} {c2:>10} {c3:>10}")

Text   : COMPUTERSCIENCECOMPUTERNETWORKCOMPUTERSCIENCE
Pattern: COMPUTER

Naive Search
Matches     : [0, 15, 30]
Comparisons : 61

KMP Search
Matches     : [0, 15, 30]
Comparisons : 45

Rabin-Karp Search
Matches     : [0, 15, 30]
Comparisons : 24

Performance Comparison
   Pattern      Naive        KMP         RK
---------------------------------------------
        AB      12467      10000       1222
       ABC      13076      10000        513
      ABCD      13245      10000        275
    ABCDAB      13306      10012        130


In [ ]:
#EXP 3
import heapq

# ---------- Union-Find for Kruskal ----------
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])   # Path Compression
        return self.parent[x]

    def union(self, x, y):
        rx = self.find(x)
        ry = self.find(y)

        if rx == ry:
            return False

        if self.rank[rx] < self.rank[ry]:
            rx, ry = ry, rx

        self.parent[ry] = rx

        if self.rank[rx] == self.rank[ry]:
            self.rank[rx] += 1

        return True


# ---------- Kruskal Algorithm ----------
def kruskal(n, edges):
    edges.sort()

    uf = UnionFind(n)
    mst = []
    total_cost = 0

    for w, u, v in edges:
        if uf.union(u, v):
            mst.append((u, v, w))
            total_cost += w

        if len(mst) == n - 1:
            break

    return mst, total_cost


# ---------- Prim Algorithm ----------
def prim(n, adj, start=0):
    visited = [False] * n
    parent = [-1] * n
    key = [float("inf")] * n

    key[start] = 0

    pq = [(0, start)]
    mst = []
    total_cost = 0

    while pq:
        weight, u = heapq.heappop(pq)

        if visited[u]:
            continue

        visited[u] = True

        if parent[u] != -1:
            mst.append((parent[u], u, weight))
            total_cost += weight

        for v, w in adj[u]:
            if not visited[v] and w < key[v]:
                key[v] = w
                parent[v] = u
                heapq.heappush(pq, (w, v))

    return mst, total_cost


# ---------- Graph Definition ----------
n = 6

edges = [
    (4, 0, 1),
    (2, 0, 2),
    (6, 1, 2),
    (5, 1, 3),
    (3, 2, 3),
    (4, 2, 4),
    (7, 3, 4),
    (6, 3, 5),
    (5, 4, 5)
]

# Create adjacency list
adj = {i: [] for i in range(n)}

for w, u, v in edges:
    adj[u].append((v, w))
    adj[v].append((u, w))


# ---------- Run Algorithms ----------
kruskal_mst, kruskal_cost = kruskal(n, edges.copy())
prim_mst, prim_cost = prim(n, adj)


# ---------- Output ----------
print("===== Kruskal's Minimum Spanning Tree =====")
for u, v, w in kruskal_mst:
    print(f"Edge ({u} - {v})  Weight = {w}")

print("Total Cost =", kruskal_cost)

print("\n===== Prim's Minimum Spanning Tree =====")
for u, v, w in prim_mst:
    print(f"Edge ({u} - {v})  Weight = {w}")

print("Total Cost =", prim_cost)

===== Kruskal's Minimum Spanning Tree =====
Edge (0 - 2)  Weight = 2
Edge (2 - 3)  Weight = 3
Edge (0 - 1)  Weight = 4
Edge (2 - 4)  Weight = 4
Edge (4 - 5)  Weight = 5
Total Cost = 18

===== Prim's Minimum Spanning Tree =====
Edge (0 - 2)  Weight = 2
Edge (2 - 3)  Weight = 3
Edge (0 - 1)  Weight = 4
Edge (2 - 4)  Weight = 4
Edge (4 - 5)  Weight = 5
Total Cost = 18


In [ ]:
#EXP 4
import heapq

# ---------- Dijkstra's Algorithm ----------
def dijkstra(graph, source):
    """
    Dijkstra's Algorithm using Min-Heap
    Time Complexity: O((V + E) log V)
    Space Complexity: O(V)
    """

    n = len(graph)
    dist = [float('inf')] * n
    prev = [None] * n

    dist[source] = 0

    pq = [(0, source)]      # (distance, vertex)
    visited = set()

    while pq:
        current_dist, u = heapq.heappop(pq)

        if u in visited:
            continue

        visited.add(u)

        for v, weight in graph[u]:
            if dist[u] + weight < dist[v]:
                dist[v] = dist[u] + weight
                prev[v] = u
                heapq.heappush(pq, (dist[v], v))

    return dist, prev


# ---------- Reconstruct Path ----------
def reconstruct_path(prev, source, target):
    path = []

    while target is not None:
        path.append(target)
        target = prev[target]

    path.reverse()

    if path and path[0] == source:
        return path

    return []


# ---------- Graph Definition ----------
graph = {
    0: [(1, 3), (2, 6)],
    1: [(2, 2), (3, 4)],
    2: [(3, 1), (4, 7)],
    3: [(4, 2), (5, 5)],
    4: [(5, 3)],
    5: []
}

source = 0

# ---------- Run Dijkstra ----------
dist, prev = dijkstra(graph, source)

# ---------- Display Results ----------
print("Shortest Paths from Source Vertex", source)
print("-" * 60)
print(f'{"Vertex":<10}{"Distance":<12}{"Path"}')
print("-" * 60)

for vertex in range(len(graph)):
    path = reconstruct_path(prev, source, vertex)
    path_str = " -> ".join(map(str, path))

    if dist[vertex] == float('inf'):
        distance = "INF"
    else:
        distance = dist[vertex]

    print(f'{vertex:<10}{distance:<12}{path_str}')

Shortest Paths from Source Vertex 0
------------------------------------------------------------
Vertex    Distance    Path
------------------------------------------------------------
0         0           0
1         3           0 -> 1
2         5           0 -> 1 -> 2
3         6           0 -> 1 -> 2 -> 3
4         8           0 -> 1 -> 2 -> 3 -> 4
5         11          0 -> 1 -> 2 -> 3 -> 5


In [ ]:
#EXP 5
import random

# Global variable to count comparisons
comparison_count = 0


# ---------- Divide and Conquer ----------
def min_max_dc(arr, low, high):
    global comparison_count

    # Only one element
    if low == high:
        return arr[low], arr[low]

    # Two elements
    if high == low + 1:
        comparison_count += 1
        if arr[low] < arr[high]:
            return arr[low], arr[high]
        else:
            return arr[high], arr[low]

    # Divide
    mid = (low + high) // 2

    left_min, left_max = min_max_dc(arr, low, mid)
    right_min, right_max = min_max_dc(arr, mid + 1, high)

    # Combine
    comparison_count += 1
    overall_min = left_min if left_min < right_min else right_min

    comparison_count += 1
    overall_max = left_max if left_max > right_max else right_max

    return overall_min, overall_max


# ---------- Naive Method ----------
def min_max_naive(arr):
    minimum = arr[0]
    maximum = arr[0]
    comparisons = 0

    for num in arr[1:]:
        comparisons += 1
        if num < minimum:
            minimum = num

        comparisons += 1
        if num > maximum:
            maximum = num

    return minimum, maximum, comparisons


# ---------- Demonstration ----------
arr = [12, 45, 7, 23, 89, 34, 2, 56, 78, 15]

comparison_count = 0
minimum, maximum = min_max_dc(arr, 0, len(arr) - 1)
dc_comparisons = comparison_count

_, _, naive_comparisons = min_max_naive(arr)

print("Array :", arr)
print("Minimum Value :", minimum)
print("Maximum Value :", maximum)
print("Divide & Conquer Comparisons :", dc_comparisons)
print("Naive Comparisons :", naive_comparisons)


# ---------- Performance Analysis ----------
print("\nPerformance Comparison")
print("-" * 60)
print(f'{"Size":>8} {"D&C":>10} {"Naive":>12} {"3n/2 - 2":>12}')
print("-" * 60)

for size in [10, 50, 100, 500]:
    arr = [random.randint(1, 1000) for _ in range(size)]

    comparison_count = 0
    minimum, maximum = min_max_dc(arr, 0, len(arr) - 1)
    dc = comparison_count

    _, _, naive = min_max_naive(arr)

    formula = (3 * size) // 2 - 2

    print(f'{size:>8} {dc:>10} {naive:>12} {formula:>12}')

Array : [12, 45, 7, 23, 89, 34, 2, 56, 78, 15]
Minimum Value : 2
Maximum Value : 89
Divide & Conquer Comparisons : 14
Naive Comparisons : 18

Performance Comparison
------------------------------------------------------------
    Size        D&C        Naive     3n/2 - 2
------------------------------------------------------------
      10         14           18           13
      50         80           98           73
     100        162          198          148
     500        754          998          748
